In [1]:
import io, os, zipfile
import numpy as np, pandas as pd, torch, torch.nn as nn
from PIL import Image
from torchvision import models, transforms
from google.colab import drive
drive.mount('/content/drive')

base = "/content/drive/MyDrive/WBC-Folder/"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
classes = ['basophil', 'eosinophil', 'erythroblast', 'ig', 'lymphocyte', 'monocyte', 'neutrophil', 'platelet']
out = base + "barcelona_forgetting_scores.csv"
assert not os.path.exists(out), "results file already exists, do not overwrite"

norm = transforms.Compose([transforms.ToTensor(),
                           transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
plain_tfm = transforms.Compose([transforms.Resize((224, 224)), norm])

def shrink_pad(img, s, size=224):
    inner = int(size * s)
    small = img.resize((inner, inner))
    arr = np.array(img)
    border = np.concatenate([arr[0], arr[-1], arr[:, 0], arr[:, -1]])
    bg = tuple(int(v) for v in np.median(border, axis=0))
    canvas = Image.new("RGB", (size, size), bg)
    canvas.paste(small, ((size - inner) // 2, (size - inner) // 2))
    return canvas

def bg_color(arr):
    flat = arr.reshape(-1, 3).astype("float32")
    lum = flat.mean(axis=1)
    return np.median(flat[lum >= np.percentile(lum, 90)], axis=0)

def bg_match(img, ref_bg):
    arr = np.array(img).astype("float32")
    gain = ref_bg / np.maximum(bg_color(arr), 1)
    return Image.fromarray(np.clip(arr * gain, 0, 255).astype("uint8"))

z = zipfile.ZipFile(base + "PBC_dataset_normal_DIB.zip")
names = {}
for n in z.namelist():
    if n.lower().endswith(".jpg") and "__MACOSX" not in n:
        names.setdefault(os.path.basename(n), []).append(n)
print("zip jpgs:", len(names), "| duplicate basenames:", sum(len(v) > 1 for v in names.values()))

refA = bg_color(np.array(Image.open(io.BytesIO(z.read(names["BNE_56697.jpg"][0]))).convert("RGB")))
print("background A:", refA.round(0))

assert os.path.exists(base + "test.csv"), "upload test.csv to WBC-Folder on Drive first"
test = pd.read_csv(base + "test.csv")
test["fname"] = test["path"].str.replace("\\", "/", regex=False).str.split("/").str[-1]
assert all(len(names.get(f, [])) == 1 for f in test["fname"]), "some test images are missing or ambiguous in the zip"
y = torch.tensor([classes.index(l) for l in test["label"]])
folder_mismatch = sum(names[f][0].split("/")[-2] != l for f, l in zip(test["fname"], test["label"]))
print(len(test), test["label"].value_counts().to_dict(), "| zip-folder vs label mismatches:", folder_mismatch)

imgs = [Image.open(io.BytesIO(z.read(names[f][0]))).convert("RGB") for f in test["fname"]]
Xplain = torch.stack([plain_tfm(im) for im in imgs])
Xfix = torch.stack([norm(shrink_pad(bg_match(im, refA), 0.65)) for im in imgs])
print("tensors:", Xplain.shape, Xfix.shape)

def load_model(p):
    m = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, len(classes))
    m.load_state_dict(torch.load(base + p, map_location=device))
    return m.to(device).eval()

def score(m, X):
    with torch.no_grad():
        return torch.cat([m(X[i:i+100].to(device)).argmax(1).cpu() for i in range(0, len(X), 100)])

rows = []
def report(name, pre, pred):
    ok = (pred == y).numpy()
    rec = {classes[c]: round(float(ok[(y == c).numpy()].mean()), 3) for c in range(len(classes))}
    print(f"{name} | {pre}: accuracy {ok.mean():.3f} | recall {rec}")
    rows.append({"model": name, "preprocessing": pre, "accuracy": round(float(ok.mean()), 4), **rec})

report("v2 (pipeline check, 0 adaptation images)", "plain", score(load_model("resnet18_wbc_v2.pt"), Xplain))

jobs = [("mll23_B (98 adaptation images)", "resnet18_wbc_v2_mll23_B.pt", ["plain", "fixes"]),
        ("raabin_fixesB (100 adaptation images)", "resnet18_wbc_v2_raabin_fixesB.pt", ["plain", "fixes"]),
        ("raabin_fixesB_40 (200 adaptation images)", "resnet18_wbc_v2_raabin_fixesB_40.pt", ["plain", "fixes"]),
        ("ku_plainA (98 adaptation images)", "resnet18_wbc_v2_ku_plainA.pt", ["plain"])]
for name, w, pres in jobs:
    m = load_model(w)
    for pre in pres:
        report(name, pre, score(m, Xplain if pre == "plain" else Xfix))

pd.DataFrame(rows).to_csv(out, index=False)
print("saved:", out)


Mounted at /content/drive
device: cuda
zip jpgs: 17093 | duplicate basenames: 0
background A: [255. 228. 203.]
2564 {'neutrophil': 500, 'eosinophil': 467, 'ig': 434, 'platelet': 352, 'erythroblast': 233, 'monocyte': 213, 'basophil': 183, 'lymphocyte': 182} | zip-folder vs label mismatches: 0
tensors: torch.Size([2564, 3, 224, 224]) torch.Size([2564, 3, 224, 224])
v2 (pipeline check, 0 adaptation images) | plain: accuracy 0.986 | recall {'basophil': 0.995, 'eosinophil': 0.998, 'erythroblast': 0.987, 'ig': 0.982, 'lymphocyte': 0.973, 'monocyte': 0.995, 'neutrophil': 0.968, 'platelet': 0.997}
mll23_B (98 adaptation images) | plain: accuracy 0.932 | recall {'basophil': 0.995, 'eosinophil': 0.938, 'erythroblast': 0.901, 'ig': 0.894, 'lymphocyte': 0.923, 'monocyte': 0.981, 'neutrophil': 0.994, 'platelet': 0.844}
mll23_B (98 adaptation images) | fixes: accuracy 0.671 | recall {'basophil': 0.885, 'eosinophil': 0.743, 'erythroblast': 0.837, 'ig': 0.654, 'lymphocyte': 0.984, 'monocyte': 0.155, '

In [2]:
import pandas as pd
r = pd.read_csv("/content/drive/MyDrive/WBC-Folder/barcelona_forgetting_scores.csv")
r["model"] = r["model"].str.replace(" adaptation images", "", regex=False)
r["drop_vs_v2"] = (0.986 - r["accuracy"]).round(3)
r = r.rename(columns={c: c[:4] for c in ['basophil','eosinophil','erythroblast','ig','lymphocyte','monocyte','neutrophil','platelet']})
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print(r.to_string(index=False))

                 model preprocessing  accuracy  baso  eosi  eryt    ig  lymp  mono  neut  plat  drop_vs_v2
v2 (pipeline check, 0)         plain    0.9860 0.995 0.998 0.987 0.982 0.973 0.995 0.968 0.997       0.000
          mll23_B (98)         plain    0.9317 0.995 0.938 0.901 0.894 0.923 0.981 0.994 0.844       0.054
          mll23_B (98)         fixes    0.6712 0.885 0.743 0.837 0.654 0.984 0.155 0.992 0.071       0.315
   raabin_fixesB (100)         plain    0.5753 0.995 0.925 0.000 0.000 0.940 0.906 0.994 0.000       0.411
   raabin_fixesB (100)         fixes    0.5164 0.869 0.752 0.000 0.000 0.951 0.667 0.998 0.000       0.470
raabin_fixesB_40 (200)         plain    0.5523 1.000 0.833 0.000 0.000 0.912 0.854 0.992 0.000       0.434
raabin_fixesB_40 (200)         fixes    0.4466 0.891 0.407 0.000 0.000 0.989 0.535 0.996 0.000       0.539
        ku_plainA (98)         plain    0.9587 0.995 0.989 0.820 0.988 0.995 0.873 0.960 0.986       0.027


In [3]:
p = "/content/drive/MyDrive/WBC-Folder/barcelona_forgetting_scores.csv"
txt = open(p).read().rstrip("\n")
print("Set-Content -Path barcelona_forgetting_results\\barcelona_forgetting_scores.csv -Encoding ascii -Value @'\n" + txt + "\n'@")

Set-Content -Path barcelona_forgetting_results\barcelona_forgetting_scores.csv -Encoding ascii -Value @'
model,preprocessing,accuracy,basophil,eosinophil,erythroblast,ig,lymphocyte,monocyte,neutrophil,platelet
"v2 (pipeline check, 0 adaptation images)",plain,0.986,0.995,0.998,0.987,0.982,0.973,0.995,0.968,0.997
mll23_B (98 adaptation images),plain,0.9317,0.995,0.938,0.901,0.894,0.923,0.981,0.994,0.844
mll23_B (98 adaptation images),fixes,0.6712,0.885,0.743,0.837,0.654,0.984,0.155,0.992,0.071
raabin_fixesB (100 adaptation images),plain,0.5753,0.995,0.925,0.0,0.0,0.94,0.906,0.994,0.0
raabin_fixesB (100 adaptation images),fixes,0.5164,0.869,0.752,0.0,0.0,0.951,0.667,0.998,0.0
raabin_fixesB_40 (200 adaptation images),plain,0.5523,1.0,0.833,0.0,0.0,0.912,0.854,0.992,0.0
raabin_fixesB_40 (200 adaptation images),fixes,0.4466,0.891,0.407,0.0,0.0,0.989,0.535,0.996,0.0
ku_plainA (98 adaptation images),plain,0.9587,0.995,0.989,0.82,0.988,0.995,0.873,0.96,0.986
'@


In [4]:
p = "/content/drive/MyDrive/WBC-Folder/barcelona_forgetting_scores.csv"
lines = open(p).read().strip().splitlines()
print("Set-Content -Path barcelona_forgetting_results\\barcelona_forgetting_scores.csv -Encoding ascii -Value @(" + ",".join("'" + l + "'" for l in lines) + ")")

Set-Content -Path barcelona_forgetting_results\barcelona_forgetting_scores.csv -Encoding ascii -Value @('model,preprocessing,accuracy,basophil,eosinophil,erythroblast,ig,lymphocyte,monocyte,neutrophil,platelet','"v2 (pipeline check, 0 adaptation images)",plain,0.986,0.995,0.998,0.987,0.982,0.973,0.995,0.968,0.997','mll23_B (98 adaptation images),plain,0.9317,0.995,0.938,0.901,0.894,0.923,0.981,0.994,0.844','mll23_B (98 adaptation images),fixes,0.6712,0.885,0.743,0.837,0.654,0.984,0.155,0.992,0.071','raabin_fixesB (100 adaptation images),plain,0.5753,0.995,0.925,0.0,0.0,0.94,0.906,0.994,0.0','raabin_fixesB (100 adaptation images),fixes,0.5164,0.869,0.752,0.0,0.0,0.951,0.667,0.998,0.0','raabin_fixesB_40 (200 adaptation images),plain,0.5523,1.0,0.833,0.0,0.0,0.912,0.854,0.992,0.0','raabin_fixesB_40 (200 adaptation images),fixes,0.4466,0.891,0.407,0.0,0.0,0.989,0.535,0.996,0.0','ku_plainA (98 adaptation images),plain,0.9587,0.995,0.989,0.82,0.988,0.995,0.873,0.96,0.986')


In [5]:
p = "/content/drive/MyDrive/WBC-Folder/barcelona_forgetting_scores.csv"
lines = open(p).read().strip().splitlines()
print("Set-Content -Path barcelona_forgetting_results\\barcelona_forgetting_scores.csv -Encoding ascii -Value @(" + ",".join("'" + l + "'" for l in lines) + ")")


Set-Content -Path barcelona_forgetting_results\barcelona_forgetting_scores.csv -Encoding ascii -Value @('model,preprocessing,accuracy,basophil,eosinophil,erythroblast,ig,lymphocyte,monocyte,neutrophil,platelet','"v2 (pipeline check, 0 adaptation images)",plain,0.986,0.995,0.998,0.987,0.982,0.973,0.995,0.968,0.997','mll23_B (98 adaptation images),plain,0.9317,0.995,0.938,0.901,0.894,0.923,0.981,0.994,0.844','mll23_B (98 adaptation images),fixes,0.6712,0.885,0.743,0.837,0.654,0.984,0.155,0.992,0.071','raabin_fixesB (100 adaptation images),plain,0.5753,0.995,0.925,0.0,0.0,0.94,0.906,0.994,0.0','raabin_fixesB (100 adaptation images),fixes,0.5164,0.869,0.752,0.0,0.0,0.951,0.667,0.998,0.0','raabin_fixesB_40 (200 adaptation images),plain,0.5523,1.0,0.833,0.0,0.0,0.912,0.854,0.992,0.0','raabin_fixesB_40 (200 adaptation images),fixes,0.4466,0.891,0.407,0.0,0.0,0.989,0.535,0.996,0.0','ku_plainA (98 adaptation images),plain,0.9587,0.995,0.989,0.82,0.988,0.995,0.873,0.96,0.986')
